In [ ]:
import dilutionUtils as dil

In [ ]:
files, injection, calibration = dil.load_config("./output/20250724_trial1_solution_injection_config.ini")

In [ ]:
injection_data = dil.get_file_paths(files['data_folder'],files['search_string'])
injection_data

In [ ]:
trial_data = dil.extract_fields(dil.read_csv(injection_data[0]))
cal_data = dil.extract_fields(dil.read_csv(files['calibration_file']))

In [ ]:
#Vadd - this variable declaration is unecessary. Just put the cal_data in the function signature
RC_ss = dil.secondary_solution_RC(injection_solution=int(calibration['volume_injection_solution sample']),Vo=int(calibration['initial_volume_secondary']))
RC = dil.calibration_RC(RC_sec=RC_ss, additions_of_injection=cal_data['Additions_of_secondary_ml'],Vc=int(cal_data['Calibration_volume_ml'][0]))

In [ ]:
slope_k = dil.calculate_k(RC_values=RC,EC_values=cal_data['Conductivity'])
print(f"{RC_ss}\n{RC}\n{slope_k}")

In [ ]:
# get the time variables background, injection, total
bg_sec = dil.time_diff_sec(start=injection['bg_start'],end=injection['bg_end'])
trial_sec = dil.time_diff_sec(start=injection['trial_start'],end=injection['trial_end'])
total_sec = dil.time_diff_sec(start=injection['bg_start'],end=injection['record_end'])


In [ ]:
bg_value = dil.get_background_ec(start_time=0, end_time=bg_sec,ec_data=trial_data['Actual Conductivity (µS/cm)'][:bg_sec])

In [ ]:
import numpy as np
from typing import Dict, List, Tuple, Union, Any

from matplotlib import pyplot as plt

In [ ]:
cumm_sec = np.arange(0,total_sec,1)
diff_sec = np.diff(cumm_sec)


In [ ]:
len(trial_data['Actual Conductivity (µS/cm)'])

In [ ]:
# needs to be fixed! this funcution requires a list of datetime objects but the arguement is a list of time strings
diff, cumm = dil.compute_time_differences_in_seconds(trial_data['Date Time'])

In [ ]:
Q = dil.calculate_Q(
    ec_bg=bg_value,
    ec_data=trial_data['Actual Conductivity (µS/cm)'],
    slope=slope_k,
    time_step=diff_sec,
    injection_volume=injection['injection_volume']
)

In [ ]:
Q

In [ ]:
def plot_injection(data:dict, 
                   time_elapsed:Union[List[float], np.ndarray],
                   bg_sec:int, trial_sec:int,
                   total_diff:Union[List[float], np.ndarray],
                   show_background=False):
    """ Plot the data of interest"""
    #ec_data = ec_data_trial_1['Actual Conductivity (µS/cm)']
    bg_value = get_background_ec(0,bg_sec,data["Actual Conductivity (µS/cm)"][bg_sec:trial_sec])
    bg_values = bg_value * np.array(total_diff)

    fig, ax = plt.subplots(1, 1, figsize=(10,6))
    if show_background==True:
        ax.plot(time_elapsed[1:], bg_values, 'ro',label='Background')

        ax.plot(time_elapsed[bg_sec:trial_sec], 
            data[bg_sec:trial_sec], 'go',
            label='Injection')
        ax.plot(time_elapsed,data["Actual Conductivity (µS/cm)"], color='blue', label='Record')
    else:
        ax.plot(time_elapsed,data["Actual Conductivity (µS/cm)"], color='blue', label='Record')

    

    # label the axes, and set the plot title
    ax.set_xlabel('Day and Time')
    ax.set_ylabel('Temp. Compenstated Conductivity [uS/cm]')
    ax.set_title('Salt Dilution Measurement')
    plt.legend()
    plt.tight_layout()

In [ ]:
dil.plot_injection(data=trial_data, time_elapsed=cumm_sec, bg_sec=bg_sec,trial_sec=trial_sec,total_diff=diff_sec,show_background=False)